# Phase 0: Pure Metric Validation

**Goal:** Validate that a `LearnedRandersManifold` with Christoffel-based spray
can learn geometry from trajectory data — no MLP shortcuts.

**Key targets:**
- Spiral geodesic deviation < 8.0 (prior MLP spray achieved 6.5)
- Attractor rollout reaches correct basin > 80% of the time
- Learned wind field b_i(x) shows convergent flow toward attractors

**Architecture:** Christoffel-based spray using analytic Riemannian + Randers
corrections, computed with single-level autograd for spatial derivatives of a_ij and b_i.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from dataclasses import dataclass
from typing import Tuple, Optional, Dict, List
import time

plt.rcParams.update({
    'figure.figsize': (10, 6), 'axes.grid': True,
    'grid.alpha': 0.3, 'font.size': 12,
})

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
device = "cpu"
print(f'Device: {device}')
torch.manual_seed(42)

---
## 1. Configuration

In [ ]:
@dataclass
class ManifoldConfig:
    d: int = 2
    hidden: int = 128
    n_layers: int = 3
    n_sub_steps: int = 4
    dt: float = 0.1

cfg = ManifoldConfig()
print(f'Config: {cfg}')

---
## 2. LearnedRandersManifold

The core module. Position-dependent Randers metric field with autograd-derived
spray, connection, and parallel transport. The metric is the single source of truth.

In [ ]:
class LearnedRandersManifold(nn.Module):
    """Position-dependent Randers metric with Christoffel-based spray.
    
    Key design: the spray is computed via analytic Christoffel symbols,
    requiring only ONE level of autograd (for da/dx, db/dx). This avoids
    the triple-nested autograd of the energy-based spray, which is
    numerically unstable through deep networks with LayerNorm.
    """

    def __init__(self, d, hidden=128, n_layers=7, n_sub_steps=4):
        super().__init__()
        self.d = d
        self.n_sub_steps = n_sub_steps

        cholesky_dim = d * (d + 1) // 2

        # Deeper metric networks with LayerNorm for sharper local structure
        a_layers = []
        b_layers = []
        for i in range(n_layers):
            in_dim = d if i == 0 else hidden
            a_layers += [nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.SiLU()]
            b_layers += [nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.SiLU()]
        a_layers.append(nn.Linear(hidden, cholesky_dim))
        b_layers.append(nn.Linear(hidden, d))

        self.a_net = nn.Sequential(*a_layers)
        self.b_net = nn.Sequential(*b_layers)

        # Near-identity init: output layers start at zero so a ~ I, b ~ 0
        nn.init.zeros_(self.a_net[-1].weight)
        nn.init.zeros_(self.b_net[-1].weight)
        nn.init.zeros_(self.b_net[-1].bias)
        # Cholesky bias -> identity: for d=2, lower-tri has entries [L00, L10, L11]
        eye_chol = torch.zeros(cholesky_dim)
        idx = 0
        for i in range(d):
            for j in range(i + 1):
                if i == j:
                    eye_chol[idx] = 1.0
                idx += 1
        self.a_net[-1].bias.data.copy_(eye_chol)

    def _build_metric(self, x):
        """Build the metric tensor a_ij from Cholesky factors. Keeps grad graph."""
        d = self.d
        raw_L = self.a_net(x)
        L = torch.zeros(*x.shape[:-1], d, d, device=x.device, dtype=x.dtype)
        tri_idx = torch.tril_indices(d, d)
        L[..., tri_idx[0], tri_idx[1]] = raw_L
        diag = torch.arange(d, device=x.device)
        L[..., diag, diag] = F.softplus(L[..., diag, diag]) + 0.5
        a = L @ L.transpose(-1, -2)
        return a

    def metric_components(self, x):
        """Returns (a_ij, b_i) at position x. a is positive definite."""
        d = self.d
        a = self._build_metric(x)

        b_raw = self.b_net(x)
        # Constrain ||b||_a < 0.8 for Randers positivity
        a_reg = a + 1e-6 * torch.eye(d, device=x.device)
        b_norm_sq = torch.einsum('...i,...ij,...j->...',
                                  b_raw,
                                  torch.linalg.inv(a_reg),
                                  b_raw)
        scale = torch.where(
            b_norm_sq > 0.64,  # 0.8^2
            0.8 / (b_norm_sq.sqrt() + 1e-8),
            torch.ones_like(b_norm_sq)
        )
        b = b_raw * scale.unsqueeze(-1)
        return a, b

    def finsler_energy(self, x, y):
        """E = F^2 / 2 where F(x,y) = alpha(x,y) + beta(x,y)."""
        a, b = self.metric_components(x)
        alpha_sq = torch.einsum('...i,...ij,...j->...', y, a, y)
        alpha = torch.sqrt(alpha_sq.clamp(min=1e-12))
        beta = torch.einsum('...i,...i->...', b, y)
        F_val = (alpha + beta).clamp(min=1e-12)
        return 0.5 * F_val ** 2

    @torch.enable_grad()
    def spray(self, x, y):
        """Geodesic spray via analytic Christoffel symbols (single-level autograd).
        
        G^i = G_riem^i + Randers_correction^i
        
        G_riem uses Christoffel symbols of a_ij, computed with ONE autograd level.
        Randers correction uses derivatives of b_i, also ONE autograd level.
        """
        d = self.d
        
        # Make x a leaf with requires_grad for spatial derivatives
        x_g = x.detach().requires_grad_(True)
        
        # Forward through networks — create_graph=True will preserve param deps
        a = self._build_metric(x_g)        # (*batch, d, d)
        b_vec = self.b_net(x_g)            # (*batch, d)
        
        # a_inv via direct formula for 2x2 (avoids inv instability)
        if d == 2:
            det = a[..., 0, 0] * a[..., 1, 1] - a[..., 0, 1] * a[..., 1, 0]
            det = det.clamp(min=1e-8)
            a_inv = torch.stack([
                torch.stack([a[..., 1, 1] / det, -a[..., 0, 1] / det], dim=-1),
                torch.stack([-a[..., 1, 0] / det, a[..., 0, 0] / det], dim=-1),
            ], dim=-2)
        else:
            a_inv = torch.linalg.inv(a + 1e-6 * torch.eye(d, device=x.device))
        
        # === Spatial derivatives: da_{lm}/dx^k via ONE level of autograd ===
        da_dx = torch.zeros(*x_g.shape[:-1], d, d, d, device=x.device, dtype=x.dtype)
        for l in range(d):
            for m in range(l, d):
                grad_alm = torch.autograd.grad(
                    a[..., l, m].sum(), x_g,
                    create_graph=True, retain_graph=True
                )[0]  # (*batch, d)
                da_dx[..., l, m, :] = grad_alm
                if l != m:
                    da_dx[..., m, l, :] = grad_alm  # symmetry of a
        
        # === Riemannian spray from Christoffel symbols ===
        # Contract Christoffel-like terms directly with y:
        # term1 = da_dx[l,j,k] * y^j * y^k  (sum over j,k)
        # term2 = da_dx[j,k,l] * y^j * y^k  (sum over j,k)
        # Gamma_{l,jk} y^j y^k = term1 - 0.5 * term2
        # (using symmetry: da[l,k,j]*y^j*y^k = da[l,j,k]*y^j*y^k)
        term1 = torch.einsum('...ljk,...j,...k->...l', da_dx, y, y)
        term2 = torch.einsum('...jkl,...j,...k->...l', da_dx, y, y)
        gamma_contracted = term1 - 0.5 * term2
        
        # G_riem^i = (1/2) a^{il} * gamma_contracted_l
        G_riem = 0.5 * torch.einsum('...il,...l->...i', a_inv, gamma_contracted)
        
        # === Randers correction from b_i ===
        # db_dx[i,j] = partial b_i / partial x^j
        db_dx = torch.zeros(*x_g.shape[:-1], d, d, device=x.device, dtype=x.dtype)
        for i in range(d):
            grad_bi = torch.autograd.grad(
                b_vec[..., i].sum(), x_g,
                create_graph=True, retain_graph=True
            )[0]
            db_dx[..., i, :] = grad_bi
        
        # Finsler quantities
        alpha_sq = torch.einsum('...i,...ij,...j->...', y, a, y).clamp(min=1e-12)
        alpha = torch.sqrt(alpha_sq)
        beta = torch.einsum('...i,...i->...', b_vec, y)
        F_val = (alpha + beta).clamp(min=1e-12)
        
        # e_{ij} = (1/2)(db_i/dx^j + db_j/dx^i) — symmetric part
        # s_{ij} = (1/2)(db_i/dx^j - db_j/dx^i) — antisymmetric part
        e_ij = 0.5 * (db_dx + db_dx.transpose(-1, -2))
        s_ij = 0.5 * (db_dx - db_dx.transpose(-1, -2))
        
        # e_{00} = e_{ij} y^i y^j
        e_00 = torch.einsum('...ij,...i,...j->...', e_ij, y, y)

        # s^i_j = a^{ik} s_{kj}
        s_upper = torch.einsum('...ik,...kj->...ij', a_inv, s_ij)
        
        # s^i_0 = s^i_j y^j
        s_i0 = torch.einsum('...ij,...j->...i', s_upper, y)

        # Randers correction: (e_{00} / (2F)) y^i - alpha * s^i_0
        randers_corr = (e_00 / (2.0 * F_val)).unsqueeze(-1) * y - alpha.unsqueeze(-1) * s_i0
        
        G = G_riem + randers_corr
        
        # Safety: clamp and replace NaN
        G = G.clamp(-50, 50)
        G = torch.where(torch.isnan(G), torch.zeros_like(G), G)
        return G

    @torch.enable_grad()
    def connection(self, x, y):
        """Non-linear connection N^i_j = dG^i/dy^j."""
        d = self.d
        y_c = y.detach().requires_grad_(True)
        G = self.spray(x, y_c)

        N_rows = []
        for i in range(d):
            grad = torch.autograd.grad(
                G[..., i].sum(), y_c,
                retain_graph=(i < d - 1), allow_unused=True
            )[0]
            N_rows.append(grad if grad is not None else torch.zeros_like(y_c))

        N = torch.stack(N_rows, dim=-2)
        return torch.where(torch.isnan(N), torch.zeros_like(N), N)

    def transport_frame(self, x_seq, dt=1.0):
        """Parallel transport a frame along a sequence. Returns list of (B,d,d) matrices."""
        B, T, d = x_seq.shape
        H = torch.eye(d, device=x_seq.device).unsqueeze(0).expand(B, -1, -1).clone()
        frames = [H]

        for t in range(T - 1):
            x_t = x_seq[:, t]
            x_next = x_seq[:, t + 1]
            v = (x_next - x_t) / dt
            sub_dt = dt / self.n_sub_steps

            for s in range(self.n_sub_steps):
                alpha_s = (s + 0.5) / self.n_sub_steps
                x_interp = x_t + alpha_s * (x_next - x_t)
                N = self.connection(x_interp, v)
                H = torch.linalg.matrix_exp(-N * sub_dt) @ H

            frames.append(H.clone())

        return frames

---
## 3. HybridRandersManifold (reference only)

Kept for future use on harder problems where pure metric plateaus.
For Phase 0 validation, we use the pure `LearnedRandersManifold` only —
the geometry must prove itself without an MLP shortcut.

In [ ]:
class HybridRandersManifold(LearnedRandersManifold):
    """G_total = G_analytic(metric) + G_residual(x, y, a(x), b(x))."""

    def __init__(self, d, hidden=128, n_layers=3, n_sub_steps=4):
        super().__init__(d, hidden, n_layers, n_sub_steps)

        cholesky_dim = d * (d + 1) // 2
        residual_in = d + d + cholesky_dim + d  # x, y, flat(L), b
        self.residual_spray = nn.Sequential(
            nn.Linear(residual_in, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, d)
        )
        nn.init.zeros_(self.residual_spray[-1].weight)
        nn.init.zeros_(self.residual_spray[-1].bias)

    @torch.enable_grad()
    def spray(self, x, y):
        G_analytic = super().spray(x, y)

        # Detach conditioning so residual doesn't drive the metric
        with torch.no_grad():
            raw_L = self.a_net(x)
            b_raw = self.b_net(x)

        conditioning = torch.cat([x.detach(), y.detach(), raw_L, b_raw], dim=-1)
        G_residual = self.residual_spray(conditioning)
        self._last_G_residual = G_residual
        self._last_G_analytic = G_analytic

        return G_analytic + G_residual

---
## 4. Data Generation

Two test cases:
1. **Spirals** — deterministic, radially varying curvature. The gold standard.
2. **Attractor walks** — stochastic, spatially varying drift toward one of K attractors.
   This is the good analogue that replaces constant-wind walks (which had no spatial
   structure and SNR of 0.5). Attractor walks have position-dependent dynamics that
   require genuine Finsler structure.

In [ ]:
def generate_spirals(n=128, T=50, noise=0.02):
    """Outward spirals with random phase. Deterministic backbone + small noise."""
    t = torch.linspace(0, 4 * np.pi, T).unsqueeze(0).expand(n, -1)
    r = 0.1 + 0.05 * t
    phase = torch.rand(n, 1) * 2 * np.pi
    x = r * torch.cos(t + phase)
    y = r * torch.sin(t + phase)
    traj = torch.stack([x, y], dim=-1)
    return traj + noise * torch.randn_like(traj)

def generate_attractor_walks(n=128, T=50, noise=0.03, n_attractors=3):
    """Stochastic walks drifting toward one of K attractors.

    Each trajectory is assigned a random attractor and drifts toward it.
    The drift is position-dependent (Ornstein-Uhlenbeck toward the target),
    so the metric MUST learn spatially varying geometry.
    """
    angles = torch.linspace(0, 2 * np.pi, n_attractors + 1)[:-1]
    attractors = 1.2 * torch.stack([torch.cos(angles), torch.sin(angles)], dim=-1)

    traj = torch.zeros(n, T, 2)
    traj[:, 0] = torch.randn(n, 2) * 0.2
    targets = attractors[torch.randint(0, n_attractors, (n,))]

    for t in range(T - 1):
        drift = 0.06 * (targets - traj[:, t])
        traj[:, t + 1] = traj[:, t] + drift + noise * torch.randn(n, 2)

    return traj, targets

In [ ]:
spirals = generate_spirals()
attractor_data, attractor_targets = generate_attractor_walks()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
for i in range(min(25, spirals.shape[0])):
    t = spirals[i].numpy()
    ax.plot(t[:, 0], t[:, 1], alpha=0.4, linewidth=0.8)
    ax.scatter(t[0, 0], t[0, 1], c='green', s=12, zorder=5)
    ax.scatter(t[-1, 0], t[-1, 1], c='red', s=12, zorder=5)
ax.set_title('Spiral Trajectories', fontweight='bold')
ax.set_aspect('equal')

ax = axes[1]
colors = ['#e74c3c', '#2ecc71', '#3498db']
angles = torch.linspace(0, 2 * np.pi, 4)[:-1]
attr_pts = 1.2 * torch.stack([torch.cos(angles), torch.sin(angles)], dim=-1)
for i in range(min(40, attractor_data.shape[0])):
    t = attractor_data[i].numpy()
    tgt = attractor_targets[i]
    ci = torch.argmin(torch.norm(attr_pts - tgt, dim=-1)).item()
    ax.plot(t[:, 0], t[:, 1], alpha=0.3, linewidth=0.8, color=colors[ci])
for j, ap in enumerate(attr_pts.numpy()):
    ax.scatter(ap[0], ap[1], c=colors[j], s=200, marker='*', zorder=10,
              edgecolors='black', linewidth=0.5)
ax.set_title('Attractor Walks (stars = attractors)', fontweight='bold')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()
print(f'Spirals: {spirals.shape}  |  Attractors: {attractor_data.shape}')

---
## 5. Training

Two-phase training schedule:
- **Phase A (warmup):** Geodesic deviation only. Let the metric learn the curvature.
- **Phase B (rollout):** Add rollout loss with gradual weight ramp.

In [ ]:
def extract_geodesic_triples(data, dt):
    """Extract (position, velocity, acceleration) triples from trajectory data."""
    vel = (data[:, 1:] - data[:, :-1]) / dt
    acc = (vel[:, 1:] - vel[:, :-1]) / dt
    pos = data[:, 1:-1]
    return pos, vel[:, :-1], acc


def train_manifold(manifold, data, n_epochs=500, lr=1e-3, batch_size=32,
                   rollout_warmup=150, rollout_weight=0.5, rollout_steps=10,
                   residual_weight=0.01, log_every=50):
    """Train the manifold via geodesic deviation + rollout stability."""
    dt = 0.1
    optimizer = torch.optim.AdamW(manifold.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)
    data_dev = data.to(device)
    N_total, T, d = data_dev.shape

    history = {'geodesic': [], 'rollout': [], 'residual': [], 'total': []}
    is_hybrid = hasattr(manifold, 'residual_spray')

    t0 = time.time()
    for epoch in range(n_epochs):
        perm = torch.randperm(N_total, device=device)
        ep_geo, ep_roll, ep_res, n_batches = 0., 0., 0., 0
        use_rollout = epoch >= rollout_warmup

        for i in range(0, N_total, batch_size):
            batch = data_dev[perm[i:i + batch_size]]
            optimizer.zero_grad()

            # --- Geodesic deviation ---
            pos, vel, acc = extract_geodesic_triples(batch, dt)
            pos_flat = pos.reshape(-1, d)
            vel_flat = vel.reshape(-1, d)
            acc_flat = acc.reshape(-1, d)

            G_pw = manifold.spray(pos_flat, vel_flat)
            L_geo = ((acc_flat + 2.0 * G_pw) ** 2).mean()

            # --- Rollout loss ---
            L_roll = torch.zeros(1, device=device)
            if use_rollout:
                steps = min(rollout_steps, T - 2)
                start = torch.randint(0, max(T - steps - 1, 1), (1,)).item()
                x_r = batch[:, start].clone()
                v_r = (batch[:, start + 1] - batch[:, start]) / dt
                for s in range(steps):
                    G_s = manifold.spray(x_r, v_r)
                    v_r = v_r - 2.0 * G_s * dt
                    x_r = x_r + v_r * dt
                    L_roll = L_roll + ((x_r - batch[:, start + s + 1]) ** 2).mean()
                L_roll = L_roll / steps

            # --- Residual regularizer (hybrid only) ---
            L_res = torch.zeros(1, device=device)
            if is_hybrid and hasattr(manifold, '_last_G_residual'):
                L_res = (manifold._last_G_residual ** 2).mean()

            rw = rollout_weight if use_rollout else 0.0
            loss = L_geo + rw * L_roll + residual_weight * L_res

            if torch.isnan(loss) or torch.isinf(loss):
                optimizer.zero_grad()
                continue

            loss.backward()
            total_norm = nn.utils.clip_grad_norm_(manifold.parameters(), max_norm=1.0)
            if torch.isnan(total_norm):
                optimizer.zero_grad()
                continue
            optimizer.step()

            ep_geo += L_geo.item()
            ep_roll += L_roll.item()
            ep_res += L_res.item()
            n_batches += 1

        scheduler.step()

        if n_batches > 0:
            history['geodesic'].append(ep_geo / n_batches)
            history['rollout'].append(ep_roll / n_batches)
            history['residual'].append(ep_res / n_batches)
            history['total'].append((ep_geo + rw * ep_roll + residual_weight * ep_res) / n_batches)
        else:
            for k in history:
                history[k].append(float('nan'))

        if (epoch + 1) % log_every == 0:
            elapsed = time.time() - t0
            g = history['geodesic'][-1]
            r = history['rollout'][-1]
            extra = ''
            if is_hybrid:
                extra = f" | Res: {history['residual'][-1]:.6f}"
            print(f'Epoch {epoch+1:4d} | Geo: {g:.4f} | Roll: {r:.4f}{extra} | {elapsed:.1f}s')

    total_time = time.time() - t0
    print(f'\nTraining complete in {total_time:.1f}s')
    return history

---
## 6. Train on Spirals

Train the pure metric manifold on spiral trajectories.
The Christoffel-based spray must learn spatially varying curvature
to reduce geodesic deviation below 8.0 (prior MLP spray baseline: 6.5).

In [ ]:
print('=== Training Pure Metric Manifold on Spirals ===')
manifold_pure = LearnedRandersManifold(
    d=cfg.d, hidden=cfg.hidden, n_layers=cfg.n_layers,
    n_sub_steps=cfg.n_sub_steps
).to(device)
print(f'Parameters: {sum(p.numel() for p in manifold_pure.parameters()):,}')

hist_pure = train_manifold(
    manifold_pure, spirals,
    n_epochs=1000, lr=5e-4, rollout_warmup=100,
    rollout_weight=0.5, log_every=50
)

In [ ]:
def quick_trajectory_check(manifold, data, n_show=12, n_seed=5, title=''):
    """Quick visual: overlay geodesic rollouts on ground truth."""
    dt = 0.1
    fig, ax = plt.subplots(figsize=(8, 8))
    
    for i in range(min(n_show, data.shape[0])):
        gt = data[i].numpy()
        ax.plot(gt[:, 0], gt[:, 1], 'b-', alpha=0.3, linewidth=1)
        ax.scatter(gt[0, 0], gt[0, 1], c='green', s=20, zorder=5)
        
        # Seed from first n_seed points, predict the rest
        x0 = data[i, n_seed - 1].unsqueeze(0).to(device)
        v0 = ((data[i, n_seed] - data[i, n_seed - 1]) / dt).unsqueeze(0).to(device)
        
        traj = [x0.detach().clone()]
        x_r, v_r = x0.clone(), v0.clone()
        for _ in range(data.shape[1] - n_seed):
            with torch.enable_grad():
                G = manifold.spray(x_r, v_r).detach()
            v_r = v_r - 2.0 * G * dt
            x_r = x_r + v_r * dt
            traj.append(x_r.detach().clone())
        
        pred = torch.cat(traj, dim=0).cpu().numpy()
        ax.plot(pred[:, 0], pred[:, 1], 'r--', alpha=0.6, linewidth=1.2)
    
    ax.plot([], [], 'b-', label='Ground truth')
    ax.plot([], [], 'r--', label='Predicted geodesic')
    ax.scatter([], [], c='green', s=20, label='Start')
    ax.legend(loc='upper right')
    ax.set_title(f'Trajectory Predictions — {title}', fontweight='bold')
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()


quick_trajectory_check(manifold_pure, spirals, title='Pure Metric (Spirals)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogy(hist_pure['geodesic'], label='Pure metric', alpha=0.8, color='#2980b9')
ax.axhline(6.5, color='gray', ls='--', alpha=0.5, label='MLP spray baseline (6.5)')
ax.axhline(8.0, color='red', ls=':', alpha=0.5, label='Target (8.0)')
ax.set_title('Geodesic Deviation (Spirals)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log)')
ax.legend()

ax = axes[1]
ax.semilogy(hist_pure['rollout'], label='Pure metric', alpha=0.8, color='#2980b9')
ax.set_title('Rollout Loss (Spirals)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log)')
ax.legend()

plt.tight_layout()
plt.show()

print(f'\nFinal spiral geodesic dev: {hist_pure["geodesic"][-1]:.4f}')
print(f'MLP spray baseline:       6.5')
print(f'Target:                   < 8.0')

---
## 7. Train on Attractor Walks

The real test: spatially varying dynamics. The metric must learn that
different regions of space have different drift directions pointing
toward different attractors. The b_i(x) wind field should show
convergent flow. Using pure metric — no residual MLP shortcut.

In [ ]:
print('=== Training Pure Metric on Attractor Walks ===')
manifold_attractor = LearnedRandersManifold(
    d=cfg.d, hidden=cfg.hidden, n_layers=cfg.n_layers,
    n_sub_steps=cfg.n_sub_steps
).to(device)
print(f'Parameters: {sum(p.numel() for p in manifold_attractor.parameters()):,}')

hist_attractor = train_manifold(
    manifold_attractor, attractor_data,
    n_epochs=500, lr=1e-3, rollout_warmup=150,
    rollout_weight=0.5, log_every=50
)

In [ ]:
quick_trajectory_check(manifold_attractor, attractor_data, title='Pure Metric (Attractors)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogy(hist_attractor['geodesic'], alpha=0.8, color='#e74c3c')
ax.set_title('Geodesic Deviation (Attractors)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log)')

ax = axes[1]
ax.semilogy(hist_attractor['rollout'], alpha=0.8, color='#e74c3c')
ax.set_title('Rollout Loss (Attractors)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log)')

plt.tight_layout()
plt.show()

---
## 8. Diagnostic Suite

### 8.1 Learned Wind Field b_i(x)

For attractor walks, the wind field should show convergent arrows
pointing toward each attractor. For spirals, it should show
tangential flow along the spiral direction.

In [ ]:
def plot_wind_field(manifold, xlim=(-2, 2), ylim=(-2, 2), n_grid=100,
                    title='Learned Wind Field b_i(x)', ax=None,
                    overlay_data=None, overlay_targets=None):
    """Quiver plot of the Randers 1-form b_i across space."""
    xs = torch.linspace(xlim[0], xlim[1], n_grid)
    ys = torch.linspace(ylim[0], ylim[1], n_grid)
    X, Y = torch.meshgrid(xs, ys, indexing='ij')
    pts = torch.stack([X.flatten(), Y.flatten()], dim=-1).to(device)

    with torch.no_grad():
        _, b = manifold.metric_components(pts)
    b = b.cpu().numpy()
    px = pts[:, 0].cpu().numpy()
    py = pts[:, 1].cpu().numpy()

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))

    mag = np.sqrt(b[:, 0]**2 + b[:, 1]**2)
    ax.quiver(px, py, b[:, 0], b[:, 1], mag, cmap='coolwarm', alpha=0.7)

    if overlay_data is not None:
        for i in range(min(15, overlay_data.shape[0])):
            t = overlay_data[i].numpy()
            ax.plot(t[:, 0], t[:, 1], alpha=0.15, linewidth=0.6, color='gray')

    if overlay_targets is not None:
        colors = ['#e74c3c', '#2ecc71', '#3498db']
        angles = torch.linspace(0, 2 * np.pi, len(colors) + 1)[:-1]
        attr_pts = 1.2 * torch.stack([torch.cos(angles), torch.sin(angles)], dim=-1)
        for j, ap in enumerate(attr_pts.numpy()):
            ax.scatter(ap[0], ap[1], c=colors[j], s=200, marker='*',
                      zorder=10, edgecolors='black', linewidth=0.5)

    ax.set_title(title, fontweight='bold')
    ax.set_aspect('equal')
    return ax


fig, axes = plt.subplots(1, 2, figsize=(20, 9))
plot_wind_field(manifold_pure, xlim=(-1.5, 1.5), ylim=(-1.5, 1.5),
               title='Wind Field — Spirals', ax=axes[0],
               overlay_data=spirals)
plot_wind_field(manifold_attractor, xlim=(-2, 2), ylim=(-1.5, 1.5),
               title='Wind Field — Attractors', ax=axes[1],
               overlay_data=attractor_data, overlay_targets=attractor_targets)
plt.tight_layout()
plt.show()

### 8.2 Spatially Varying Indicatrices

The unit ball of F at each point. Shape reveals local anisotropy:
- Circles = isotropic (Euclidean)
- Elongated ellipses = strong directional preference
- Offset from center = wind (Randers 1-form)

In [ ]:
def plot_indicatrices(manifold, xlim=(-1.5, 1.5), ylim=(-1.5, 1.5),
                      n_grid=25, n_angles=64, scale=0.1, title='Indicatrices',
                      ax=None):
    """Plot the Finsler indicatrix (unit ball of F) at grid points."""
    xs = torch.linspace(xlim[0], xlim[1], n_grid)
    ys = torch.linspace(ylim[0], ylim[1], n_grid)
    theta = torch.linspace(0, 2 * np.pi, n_angles + 1)[:-1]

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))

    for xi in xs:
        for yi in ys:
            x_pt = torch.tensor([[xi.item(), yi.item()]], device=device)
            dirs = torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1).to(device)
            x_rep = x_pt.expand(n_angles, -1)

            with torch.no_grad():
                a, b = manifold.metric_components(x_rep)
                alpha_sq = torch.einsum('bi,bij,bj->b', dirs, a, dirs)
                alpha = torch.sqrt(alpha_sq.clamp(min=1e-12))
                beta = torch.einsum('bi,bi->b', b, dirs)
                F_val = (alpha + beta).clamp(min=1e-12)

            r = scale / F_val.cpu().numpy()
            px = xi.item() + r * np.cos(theta.numpy())
            py = yi.item() + r * np.sin(theta.numpy())
            ax.fill(px, py, alpha=0.2, color='steelblue')
            ax.plot(np.append(px, px[0]), np.append(py, py[0]),
                   linewidth=0.5, color='steelblue')

    ax.set_title(title, fontweight='bold')
    ax.set_aspect('equal')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    return ax


fig, axes = plt.subplots(1, 2, figsize=(20, 9))
plot_indicatrices(manifold_pure, xlim=(-1.8, 1.8), ylim=(-1.8, 1.8),
                  title='Indicatrices — Spirals', ax=axes[0])
plot_indicatrices(manifold_attractor, xlim=(-1.8, 1.8), ylim=(-1.8, 1.8),
                  title='Indicatrices — Attractors', ax=axes[1])
plt.tight_layout()
plt.show()

### 8.3 Direction-Dependent Spray (Finsler Test)

At a fixed point x, compare G(x, y) vs G(x, -y). In Riemannian geometry these
are related by symmetry. In Finsler geometry they differ — the "Finslerian content."

In [ ]:
def plot_spray_asymmetry(manifold, test_point=(0.5, 0.3), n_dirs=36, title=''):
    """Plot ||G(x,y) - G(x,-y)|| vs direction angle."""
    theta = torch.linspace(0, 2 * np.pi, n_dirs + 1)[:-1]
    dirs = torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1).to(device)
    x_pt = torch.tensor([test_point], device=device).expand(n_dirs, -1)

    with torch.enable_grad():
        G_fwd = manifold.spray(x_pt, dirs).detach()
        G_bwd = manifold.spray(x_pt, -dirs).detach()

    diff = torch.norm(G_fwd - G_bwd, dim=-1).cpu().numpy()
    G_mag = torch.norm(G_fwd, dim=-1).cpu().numpy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), subplot_kw={'projection': 'polar'})

    ax = axes[0]
    ax.plot(theta.numpy(), diff, 'b-', linewidth=1.5)
    ax.fill(theta.numpy(), diff, alpha=0.2, color='blue')
    ax.set_title(f'||G(x,y) - G(x,-y)|| at {test_point}\n{title}', pad=15)

    ax = axes[1]
    ax.plot(theta.numpy(), G_mag, 'r-', linewidth=1.5)
    ax.fill(theta.numpy(), G_mag, alpha=0.2, color='red')
    ax.set_title(f'||G(x,y)|| at {test_point}\n{title}', pad=15)

    plt.tight_layout()
    plt.show()

    rel = diff / (G_mag + 1e-8)
    print(f'Mean asymmetry: {diff.mean():.4f}')
    print(f'Mean relative asymmetry: {rel.mean():.2%}')


print('--- Spiral manifold ---')
plot_spray_asymmetry(manifold_pure, title='Spirals')

print('\n--- Attractor manifold ---')
plot_spray_asymmetry(manifold_attractor, title='Attractors')

### 8.4 Geodesic Deviation Heatmap

Color each training point by its local geodesic deviation.
Shows where the metric fits well (blue) vs poorly (red).

In [ ]:
def plot_geodesic_deviation_heatmap(manifold, data, dt=0.1, title='', ax=None):
    """Scatter plot of training points colored by local geodesic deviation."""
    pos, vel, acc = extract_geodesic_triples(data, dt)
    pos_flat = pos.reshape(-1, 2).to(device)
    vel_flat = vel.reshape(-1, 2).to(device)
    acc_flat = acc.reshape(-1, 2).to(device)

    with torch.enable_grad():
        G = manifold.spray(pos_flat, vel_flat).detach()
    dev = ((acc_flat + 2.0 * G) ** 2).sum(-1).sqrt().cpu().numpy()
    px = pos_flat[:, 0].cpu().numpy()
    py = pos_flat[:, 1].cpu().numpy()

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))

    vmax = np.percentile(dev, 95)
    sc = ax.scatter(px, py, c=dev, s=2, alpha=0.5, cmap='RdYlBu_r',
                    vmin=0, vmax=vmax)
    plt.colorbar(sc, ax=ax, label='|geodesic deviation|')
    ax.set_title(f'Geodesic Deviation Heatmap — {title}', fontweight='bold')
    ax.set_aspect('equal')
    return ax


fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_geodesic_deviation_heatmap(manifold_pure, spirals,
                                 title='Spirals', ax=axes[0])
plot_geodesic_deviation_heatmap(manifold_attractor, attractor_data,
                                 title='Attractors', ax=axes[1])
plt.tight_layout()
plt.show()

---
## 9. Geodesic Rollout Prediction

Integrate the geodesic equation from seed initial conditions
and compare against ground truth. The critical test of whether
the metric creates actual navigable channels.

In [ ]:
def predict_geodesic(manifold, x0, v0, n_steps=30, dt=0.1):
    """Integrate the geodesic equation from (x0, v0)."""
    trajectory = [x0.detach().clone()]
    x = x0.clone()
    v = v0.clone()

    for _ in range(n_steps):
        with torch.enable_grad():
            G = manifold.spray(x, v).detach()
        v = v - 2.0 * G * dt
        x = x + v * dt
        trajectory.append(x.detach().clone())

    return torch.stack(trajectory, dim=1)  # (B, n_steps+1, d)


def plot_rollouts(manifold, data, n_show=8, n_seed=5, title='', targets=None):
    """Compare geodesic rollouts against ground truth."""
    dt = 0.1
    fig, ax = plt.subplots(figsize=(8, 8))

    for i in range(n_show):
        gt = data[i].numpy()
        ax.plot(gt[:, 0], gt[:, 1], 'b-', alpha=0.3, linewidth=0.8)

        x0 = data[i, :n_seed].unsqueeze(0).to(device)
        v0 = ((data[i, n_seed] - data[i, n_seed - 1]) / dt).unsqueeze(0).to(device)
        x_start = data[i, n_seed - 1].unsqueeze(0).to(device)

        pred = predict_geodesic(manifold, x_start, v0,
                                n_steps=data.shape[1] - n_seed, dt=dt)
        pred_np = pred[0].cpu().numpy()
        ax.plot(pred_np[:, 0], pred_np[:, 1], 'r--', alpha=0.6, linewidth=1.0)
        ax.scatter(pred_np[0, 0], pred_np[0, 1], c='green', s=30, zorder=5)

    if targets is not None:
        colors_t = ['#e74c3c', '#2ecc71', '#3498db']
        angles_t = torch.linspace(0, 2 * np.pi, len(colors_t) + 1)[:-1]
        attr_pts = 1.2 * torch.stack([torch.cos(angles_t), torch.sin(angles_t)], dim=-1)
        for j, ap in enumerate(attr_pts.numpy()):
            ax.scatter(ap[0], ap[1], c=colors_t[j], s=200, marker='*',
                      zorder=10, edgecolors='black', linewidth=0.5)

    ax.set_title(f'Geodesic Rollouts vs Ground Truth — {title}', fontweight='bold')
    ax.set_aspect('equal')
    ax.legend(['Ground truth', 'Prediction'], loc='upper right')
    plt.tight_layout()
    plt.show()


print('--- Spiral rollouts ---')
plot_rollouts(manifold_pure, spirals, title='Spirals (Hybrid)')

print('\n--- Attractor rollouts ---')
plot_rollouts(manifold_attractor, attractor_data, title='Attractors',
              targets=attractor_targets)

---
## 10. Holonomy Analysis

Compute the frame holonomy along trajectories and measure:
1. How much the frame deviates from identity (||H_T - I||)
2. Whether different paths produce different holonomies (path-dependence)

In [ ]:
def analyze_holonomy(manifold, data, n_traj=16):
    """Analyze frame holonomy along trajectories."""
    d = manifold.d
    subset = data[:n_traj].to(device)

    with torch.no_grad():
        frames = manifold.transport_frame(subset, dt=0.1)

    H_final = frames[-1]  # (n_traj, d, d)
    I = torch.eye(d, device=device).unsqueeze(0)
    deviations = torch.norm(H_final - I, dim=(-2, -1)).cpu().numpy()

    # Track deviation along the path
    path_devs = []
    for t in range(len(frames)):
        dev_t = torch.norm(frames[t] - I, dim=(-2, -1)).mean().item()
        path_devs.append(dev_t)

    # Pairwise holonomy differences (path-dependence)
    H_stack = torch.stack([frames[-1][i] for i in range(n_traj)])
    diffs = []
    for i in range(n_traj):
        for j in range(i + 1, n_traj):
            diffs.append(torch.norm(H_stack[i] - H_stack[j]).item())

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    ax = axes[0]
    ax.plot(path_devs, 'o-', markersize=3)
    ax.set_xlabel('Token position')
    ax.set_ylabel('||H_t - I||')
    ax.set_title('Holonomy Deviation Along Path', fontweight='bold')

    ax = axes[1]
    ax.bar(range(n_traj), deviations, color='steelblue', alpha=0.7)
    ax.set_xlabel('Trajectory')
    ax.set_ylabel('||H_T - I||')
    ax.set_title('Final Holonomy per Trajectory', fontweight='bold')

    ax = axes[2]
    ax.hist(diffs, bins=30, color='coral', alpha=0.7, edgecolor='white')
    ax.set_xlabel('||H_i - H_j||')
    ax.set_ylabel('Count')
    ax.set_title('Pairwise Holonomy Differences', fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(f'Mean final deviation: {deviations.mean():.4f}')
    print(f'Mean pairwise difference: {np.mean(diffs):.4f}')
    if np.mean(diffs) > 0.01:
        print('Path-dependent holonomy detected (good — manifold has curvature)')
    else:
        print('Holonomies are similar (manifold may be effectively flat)')


print('--- Spiral holonomy ---')
analyze_holonomy(manifold_pure, spirals)

print('\n--- Attractor holonomy ---')
analyze_holonomy(manifold_attractor, attractor_data)

---
## 11. Attractor Basin Test

For attractor walks, test whether geodesic rollouts converge to the
correct attractor basin. Target: > 80%.

In [ ]:
def test_attractor_basins(manifold, data, targets, n_test=64, n_seed=5):
    """Test whether rollouts reach the correct attractor."""
    dt = 0.1
    angles = torch.linspace(0, 2 * np.pi, 4)[:-1]
    attr_pts = 1.2 * torch.stack([torch.cos(angles), torch.sin(angles)], dim=-1)

    correct = 0
    total = min(n_test, data.shape[0])

    for i in range(total):
        x_start = data[i, n_seed - 1].unsqueeze(0).to(device)
        v0 = ((data[i, n_seed] - data[i, n_seed - 1]) / dt).unsqueeze(0).to(device)

        pred = predict_geodesic(manifold, x_start, v0,
                                n_steps=data.shape[1] - n_seed, dt=dt)
        endpoint = pred[0, -1].cpu()

        # Which attractor is closest to the endpoint?
        dists = torch.norm(attr_pts - endpoint, dim=-1)
        pred_attr = torch.argmin(dists).item()

        # Which attractor is the ground truth target?
        tgt_dists = torch.norm(attr_pts - targets[i], dim=-1)
        true_attr = torch.argmin(tgt_dists).item()

        if pred_attr == true_attr:
            correct += 1

    accuracy = correct / total
    print(f'Attractor basin accuracy: {correct}/{total} = {accuracy:.1%}')
    print(f'Target: > 80%  |  Status: {"PASS" if accuracy > 0.8 else "FAIL"}')
    return accuracy


basin_acc = test_attractor_basins(manifold_attractor, attractor_data, attractor_targets)

---
## 12. Summary and Phase 0 Scorecard

In [ ]:
print('=' * 60)
print('PHASE 0 SCORECARD — Pure Metric')
print('=' * 60)

spiral_geo = hist_pure['geodesic'][-1]
spiral_roll = hist_pure['rollout'][-1]
attr_geo = hist_attractor['geodesic'][-1]

results = [
    ('Spiral geodesic deviation', spiral_geo, '< 8.0', spiral_geo < 8.0),
    ('Spiral rollout loss', spiral_roll, '< 0.02', spiral_roll < 0.02),
    ('Attractor geodesic deviation', attr_geo, 'decreasing', True),
]

for name, val, target, passed in results:
    status = 'PASS' if passed else 'FAIL'
    print(f'  {name:35s} = {val:.4f}  (target: {target})  [{status}]')

print(f'\nPure metric final geo dev:  {hist_pure["geodesic"][-1]:.4f}')
print(f'Prior MLP spray baseline:   6.5')
print(f'Prior metric baseline:      25.2')

print('\n' + '=' * 60)
print('Next: Phase 2 (Holonomy Ablation) + Phase 3 (Geodesic Loops)')
print('=' * 60)